# 04.2 · q-transfer FAILURE analysis — single examples

Near-identical to `04_q_transfer.ipynb`, but instead of generating fresh setups it **reconstructs instances that FAILED causally in the completed run** (`results/diff/{q_transfer,filter_transfer}`) and re-runs the same intervention + eval on them — one at a time, for debugging/finetuning. Prompts go through the current (fixed) `materialize` so the pivot is a real token; gaze uses the new absolute `gaze(SRC) > gaze(DST)` metric with the clean baseline.

_A scaled-up version comes later; for now these are single hand-picked failures._

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2, 3")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP
PATCH_HEADS = GOAL_HEADS                    # heads carrying the transplant (GOAL | UNION | ...)
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape
from src.utils.experiment_utils import set_seed

def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift

def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)),
                     title=f"[bold]{title}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

2026-09-03 15:51:43 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Load failure cases from the completed run

`rebuild()` reconstructs the data instance from a saved record (q from `src_progress + [expected_next]`, filter from the record key). `load_fails` keeps only causal failures; we prioritize **readout-shifted** failures (`readout.patched == src` but the generation didn't follow) — the most diagnostic 'the model knew but didn't act' cases.

In [16]:
import json, glob
from pathlib import Path
from src.utils import env_utils

MODEL_SHORT = "Qwen2.5-Coder-32B-Instruct"
Q_ROOT = Path(env_utils.DEFAULT_RESULTS_DIR) / "diff" / MODEL_SHORT / "q_transfer" / "multi_pred"
F_ROOT = Path(env_utils.DEFAULT_RESULTS_DIR) / "diff" / MODEL_SHORT / "q_transfer" / "filter"

def rebuild(r):
    """Reconstruct the data instance object from one saved record."""
    dt = r["datatype"]
    if r.get("setup", "q") == "filter":
        ft = D.FILTER_TASKS[dt]
        kv = dict(p.split("=", 1) for p in r["key"].split("|") if "=" in p)
        return D.FilterTransferInstance(
            task=ft, pair=tuple(kv["pair"].split("-")), var_name=ft.var_names[0],
            src_lang=r["src_lang"], dst_lang=r["dst_lang"], src_fill=kv["srcfill"], dst_fill=kv["dstfill"],
            expected_next=kv["dstfill"], postfix=r["naming"], meta={"datatype": dt, "dst_next": kv["srcfill"]})
    bank = D.QTASKS[dt]; by = {p.id: p for p in bank.predicates}
    ids = list(r["src_progress"]) + [r["expected_next"]]
    task = D.dataclasses.replace(bank, id=f"{bank.id}_k{len(ids)}", predicates=tuple(by[i] for i in ids))
    return D.QTransferInstance(
        task=task, var_name=bank.var_names[0], presentation=r["presentation"], src_lang=r["src_lang"],
        dst_lang=r["dst_lang"], src_progress=list(r["src_progress"]), dst_progress=list(r["dst_progress"]),
        expected_next=r["expected_next"], naming=r["naming"] if r["naming"] in ("natural", "letters") else "natural",
        meta={"datatype": dt, "dst_next": r["dst_next"]})

def load_fails(root):
    out = []
    for f in glob.glob(f"{root}/*/*/results.json"):
        for r in json.loads(Path(f).read_text()):
            if r.get("causal") and not r["causal"]["triggered"]:
                r["_cfg"] = Path(f).parent.name; out.append(r)
    return out

def show_fails(fails, k=12):
    ro_shift = [r for r in fails if (r.get("readout") or {}).get("patched") in ("src", "both")]
    print(f"{len(fails)} causal failures  ({len(ro_shift)} readout-shifted — prioritized)")
    for r in ro_shift[:k]:
        ro = r.get("readout") or {}
        print(f"  [{r['_cfg']}] SRC {r['expected_next']!r} -> DST {r['dst_next']!r} | "
              f"causal={r['causal']['label']} · readout_patched={ro.get('patched')}")
    return ro_shift or fails

def pick_failure(fails, i=0):
    """Deterministic pick — readout-shifted failures first, then the i-th."""
    ro_shift = [r for r in fails if (r.get("readout") or {}).get("patched") in ("src", "both")]
    pool = ro_shift or fails
    return pool[i % len(pool)]


def make_filter(datatype, target, frontier, src_lang="python", dst_lang="python", postfix="targeted"):
    """Construct the SPECIFIC composed-filter case `target<-frontier` (matches the scale-up `pair` column):
    SRC is about to apply `target` (=expected_next=dst_fill), clean DST applies `frontier`
    (=dst_next=src_fill). No sampling / no results dependency. Asserts the pair is composable."""
    ft = D.FILTER_TASKS[datatype]
    assert D.filter_compatible(target, frontier), f"{target}/{frontier} is an incompatible pair"
    return D.FilterTransferInstance(
        task=ft, pair=(frontier, target), var_name=ft.var_names[0], src_lang=src_lang, dst_lang=dst_lang,
        src_fill=frontier, dst_fill=target, expected_next=target,
        postfix=postfix, meta={"datatype": datatype, "dst_next": frontier})


def pick_by_pair(fails, target, frontier):
    """The recorded failure whose concept pair is `target<-frontier` (expected_next<-dst_next), or None."""
    hits = [r for r in fails if r["expected_next"] == target and r["dst_next"] == frontier]
    return hits[0] if hits else None

qfails = load_fails(Q_ROOT)
qcands = show_fails(qfails)

172 causal failures  (141 readout-shifted — prioritized)
  [q__int__randnm__python-cpp] SRC 'absolute_value' -> DST 'digit_count' | causal=dst · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'last_digit' -> DST 'is_positive' | causal=dst · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'absolute_value' -> DST 'is_positive' | causal=dst · readout_patched=both
  [q__int__randnm__python-cpp] SRC 'is_prime' -> DST 'is_negative' | causal=dst · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'is_perfect_square' -> DST 'is_palindrome' | causal=neither · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'sign' -> DST 'is_negative' | causal=dst · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'sign' -> DST 'is_positive' | causal=dst · readout_patched=both
  [q__int__randnm__python-cpp] SRC 'is_positive' -> DST 'is_even' | causal=dst · readout_patched=src
  [q__int__randnm__python-cpp] SRC 'is_multiple_of_3' -> DST 'is_negative' | causal=dst · readout_patched

## Render one q-transfer failure — SRC and DST prompts with pivots

In [44]:
qpick = pick_failure(qfails, i=13)          # <- change i to inspect a different failure
inst = rebuild(qpick)
m = D.materialize(inst)                     # current (fixed) chat template
src_prompt, dst_prompt = m["src_prompt"], m["dst_prompt"]
# src_prompt, dst_prompt = dst_prompt, src_prompt  # swap to show the DST first (the one that failed)

piv_src, piv_dst = pivot_of(src_prompt), pivot_of(dst_prompt)
print(f"[{qpick['_cfg']}]  expected_next = {inst.expected_next!r}  ({inst.expected_next_desc})")
print(f"OLD run verdict: causal={qpick['causal']['label']} (FAILED) · "
      f"readout {qpick.get('readout', {}).get('source')}->{qpick.get('readout', {}).get('dest')}->"
      f"{qpick.get('readout', {}).get('patched')}")
show(src_prompt, f"SRC · progress={inst.src_progress} · about to write {inst.expected_next!r} · pivot@{piv_src}", piv=-1)
show(dst_prompt, f"DST · progress={inst.dst_progress} · pivot@{piv_dst}", piv=-1)

[q__int__randnm__python-cpp]  expected_next = 'is_multiple_of_3'  (whether the number is a multiple of three)
OLD run verdict: causal=dst (FAILED) · readout src->dst->src


╭─ SRC · progress=['is_palindrome', 'digit_sum'] · about to write 'is_multiple_of_3' · pivot@138 ─────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given an integer `n`, computes:\n               │
│ (1) whether the number reads the same forwards and backwards\n                                                  │
│ (2) the sum of the digits\n                                                                                     │
│ (3) whether the number is a multiple of three\n                                                                 │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(n: int):\n                                                                                             │
│     a = str(abs(n)) == str(abs(n))[::-1]\n                                                                      │
│     b = sum(int(d) for d in str(abs(n)))\n                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · progress=['is_palindrome'] · pivot@159 ──────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given an integer `n`, computes:\n                  │
│ (1) whether the number reads the same forwards and backwards\n                                                  │
│ (2) the sum of the digits\n                                                                                     │
│ (3) whether the number is a multiple of three\n                                                                 │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(int n){\n                                                                                             │
│     string a_s = to_string(abs(n));\n                                                                           │
│     string a_r = a_s;\n                                                                                         │
│     reverse(a_r.begin(), a_r.end());\n                                                                          │
│     bool a = (a_s == a_r);\n                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Intervention — transplant the SRC pivot query into DST

Cache the (pre-RoPE) query of `PATCH_HEADS` at the SRC pivot; inject it at the DST pivot (prefill + re-clamped for `n_steer` decode steps). Expect the patched DST to implement `expected_next` instead of its own next predicate.

In [45]:
# ── intervention: transplant the SRC pivot query into DST — grouped per run ──
from src.attention import get_attention_matrices
Q_LAYERS = sorted({l for l, _ in PATCH_HEADS})
q_src = cache_src_proj(src_prompt, [piv_src], Q_LAYERS, "q")

def q_inject_specs(qcache, dest_pivot, heads=PATCH_HEADS):
    qh = heads_per_layer(heads, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],   # prefill
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])            # decode re-clamp

src_tok = prepare_input(prompts=src_prompt, tokenizer=mt)
dst_tok = prepare_input(prompts=dst_prompt, tokenizer=mt)
pf, dc  = q_inject_specs(q_src, piv_dst)

# clean baseline attn over the TASK (numbered predicate list); query = pivot (last token)
_spans    = [token_span(dst_prompt, p.description) for p in inst.task.predicates]
TASK_POS  = list(range(min(s[0] for s in _spans), max(s[1] for s in _spans)))
task_toks = [mt.tokenizer.decode([int(dst_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_i  = get_attention_matrices(dst_tok, mt, value_weighted=True)
def _cattn(h): return pivot_attn(_clean_i, list(h), TASK_POS, piv_dst)

# ── source_run — clean SRC (frontier = expected_next); generation + goalscope ──
gen_src = greedy(src_tok, max_new=96)
rich_print(f"[red1]source_run · clean SRC (frontier {inst.expected_next!r}):[/red1] \n{escape(gen_src)}")
ro_src = goalscope_readout("   ↳ goalscope · source_run", src_prompt, piv_src, heads=PATCH_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST (frontier = dst_next); generation + goalscope + ATTN_VIS baseline ──
gen_dst = greedy(dst_tok, max_new=96)
rich_print(f"[sea_green3]destination_run · clean DST (frontier {inst.meta['dst_next']!r}):[/sea_green3] \n{escape(gen_dst)}")
ro_dst = goalscope_readout("   ↳ goalscope · destination_run", dst_prompt, piv_dst, heads=PATCH_HEADS)[0]["patched_gen"]
attn_clean_goal = _cattn(GOAL_HEADS)      # clean baseline for the gaze eval
show_delta_tokens(task_toks, _cattn(PATCH_HEADS), "ATTN_VIS · pivot→TASK (clean · PATCH_HEADS)")
# show_delta_tokens(task_toks, _cattn(GOAL_HEADS),  "ATTN_VIS · pivot→TASK (clean · GOAL_HEADS)")

# ── patched_run — SRC query injected into DST (expect expected_next); gen + goalscope + ATTN_DIFF ──
_iface = make_patch_interface(mt, pf)
gen_patched = greedy(dst_tok, prefill_specs=pf, decode_specs=dc, n_steer=32, max_new=96)
rich_print(f"[salmon1]patched_run · DST (expect {inst.expected_next!r}):[/salmon1] \n{escape(gen_patched)}")
ro_patched = goalscope_readout("   ↳ goalscope · patched_run", dst_prompt, piv_dst, heads=PATCH_HEADS, iface=_iface)[0]["patched_gen"]
_patched_i = get_attention_matrices(dst_tok, mt, value_weighted=True, patch_interface=_iface)
attn_patched_goal = pivot_attn(_patched_i, list(GOAL_HEADS), TASK_POS, piv_dst)   # patched, for gaze
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(PATCH_HEADS), TASK_POS, piv_dst) - _cattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(task_toks, attn_patched_goal, "ATTN_VIS · patched (GOAL_HEADS)")

source_run · clean SRC (frontier 'is_multiple_of_3'): 
 c = n % 3 == 0
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=138 '    ' => I'm checking if the variable is divisible by 3.

destination_run · clean DST (frontier 'digit_sum'): 
 int sum = 0;
    while(n){
        sum += n % 10;
        n /= 10;
    }
    bool b = (sum % 3 == 0);
    return make_tuple(a, sum, b);
}
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination_run  pos=159 '    ' => I'm calculating the sum of the digits of the number.

... (2 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=159 '    ' => I'm checking if the variable is divisible by 3.

ATTN_DIFF · patched−clean (PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=1.079)

ATTN_VIS · patched (GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=1.368)

In [47]:
# ── DEBUG VARIANTS on the q-transfer causal intervention (mirror of the composed-filter debug cell) ──
# Reuses src_prompt/dst_prompt, piv_src/piv_dst, src_tok/dst_tok, PATCH_HEADS, pf/dc from the cell above.
# (1) boost the PATCH_HEADS OV write by ALPHA after the q-patch; (2) transplant the SRC query over the last
# N_PIVOT pivot tokens on the PREFILL; decode still steers the last token only (n_steer). NOTE: unlike the
# targeted-filter case, a q-transfer SRC/DST differ in their LAST prefilled line, so the last-N tokens are
# NOT identical here (the sanity line reports it) — the window aligns by offset-from-pivot regardless.
ALPHA   = 2.0
N_PIVOT = 5
N_STEER = 32

rich_print(f"[grey62]baseline · single-pivot q-patch:[/grey62] \n{escape(gen_patched)}")

def make_boosted_interface(mt, patch_specs, boost_heads, alpha, positions=(-1,)):
    def iface(mt, inputs, patches=None, model_kwargs=None):
        with P.proj_patch_hooks(mt, patch_specs), P.boost_head_ov(mt, boost_heads, alpha, positions=positions):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return iface

def q_inject_specs_multi(src_prompt, src_pivot, dst_pivot, heads, n_pivot):
    qh = P.heads_per_layer(heads, "q")
    q_layers = sorted({l for l, _ in heads})
    src_win = list(range(src_pivot - n_pivot + 1, src_pivot + 1))   # cache row r -> dst_win[r]
    dst_win = list(range(dst_pivot - n_pivot + 1, dst_pivot + 1))
    qcache  = cache_src_proj(src_prompt, src_win, q_layers, "q")     # (n_pivot, width) per layer
    prefill = [dict(which="q", cache=qcache, heads=qh, positions=dst_win)]
    dcache  = {l: qcache[l][-1:] for l in qcache}                    # pivot only (last row) for decode
    decode  = [dict(which="q", cache=dcache, heads=qh, positions=[-1])]
    return prefill, decode

# ── (1) boost PATCH_HEADS OV contribution after the q-patch ──────────────────────
with P.boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)):
    gen_boost = greedy(dst_tok, prefill_specs=pf, decode_specs=dc, n_steer=N_STEER, max_new=96)
rich_print(f"[salmon1]patched + OV-boost(α={ALPHA}) · expect {inst.expected_next!r}:[/salmon1] \n{escape(gen_boost)}")
_iface_boost = make_boosted_interface(mt, pf, PATCH_HEADS, ALPHA)
ro_boost = goalscope_readout("   ↳ goalscope · patched+boost", dst_prompt, piv_dst, heads=PATCH_HEADS, iface=_iface_boost)[0]["patched_gen"]

# # ── (2) transplant SRC Q over the last N_PIVOT pivot tokens (prefill); decode still patches -1 ──
# _sw = [mt.tokenizer.decode([int(src_tok["input_ids"][0][piv_src - N_PIVOT + 1 + i])]) for i in range(N_PIVOT)]
# _dw = [mt.tokenizer.decode([int(dst_tok["input_ids"][0][piv_dst - N_PIVOT + 1 + i])]) for i in range(N_PIVOT)]
# print(f"pivot window (N_PIVOT={N_PIVOT})  SRC {_sw}  ->  DST {_dw}  {'✓ aligned' if _sw == _dw else '✗ differ (q-transfer)'}")

# pf_m, dc_m = q_inject_specs_multi(src_prompt, piv_src, piv_dst, PATCH_HEADS, N_PIVOT)
# gen_multi = greedy(dst_tok, prefill_specs=pf_m, decode_specs=dc_m, n_steer=N_STEER, max_new=96)
# rich_print(f"[salmon1]patched · Q over last {N_PIVOT} pivots · expect {inst.expected_next!r}:[/salmon1] \n{escape(gen_multi)}")
# _iface_multi = make_patch_interface(mt, pf_m)
# ro_multi = goalscope_readout("   ↳ goalscope · multi-pivot", dst_prompt, piv_dst, heads=PATCH_HEADS, iface=_iface_multi)[0]["patched_gen"]

# # ── (1)+(2) combined — multi-pivot Q transplant + OV boost ──────────────────────
# with P.boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)):
#     gen_both = greedy(dst_tok, prefill_specs=pf_m, decode_specs=dc_m, n_steer=N_STEER, max_new=96)
# rich_print(f"[salmon1]patched · multi-pivot + OV-boost(α={ALPHA}):[/salmon1] \n{escape(gen_both)}")

baseline · single-pivot q-patch: 
 int sum = 0;
    for (char c : a_s) {
        sum += c - '0';
    }
    bool b = (sum % 3 == 0);
    return make_tuple(a, sum, b);
}
</code>

patched + OV-boost(α=2.0) · expect 'is_multiple_of_3': 
 bool b = (n % 3 == 0);
    return make_tuple(a, b);
}
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched+boost  pos=159 '    ' => I'm checking if the variable is divisible by 3.

---
## Evaluate the intervention (`src.diff.eval`)

Three complementary evals of the q-transfer intervention (single source of truth in `src/diff/eval.py`):

1. **Causal** (what we care about) — oracle judges whether the patched continuation pursues the SRC target (or a non-trivial part).
2. **Readout shift** — classifies all three goalscope readouts (source/destination/patched); a case is *valid* only if the clean readouts land as expected (source→SRC, dest→DST), and *shifted* if the patched readout expresses the SRC target.
3. **Gaze** — rule-based: did the value-weighted attention delta move toward the SRC-target tokens vs the DST-target tokens?

The causal & readout evals call the Claude oracle (`ask_claude`, disk-cached). The same `eval_q_transfer` works on the composed-filter instance below.

In [ ]:
from src.diff import eval as E

qeval = E.eval_q_transfer(
    inst,
    # patched_gen=gen_patched,                                   # 1. causal (oracle)
    patched_gen=gen_boost,
    readouts=(ro_src, ro_dst, ro_patched),                    # 2. readout shift (oracle)
    gaze=dict(prompt=dst_prompt, task_positions=TASK_POS,     # 3. gaze (rule-based, absolute)
              attn_patched=attn_patched_goal, attn_clean=attn_clean_goal),
)
rich_print(f"[bold]EVAL[/bold] · expected_next={inst.expected_next!r}\n{qeval.summary()}")
rich_print("  causal :", qeval.causal)
rich_print("  readout:", qeval.readout)
rich_print("  gaze   :", qeval.gaze)

EVAL · expected_next='is_multiple_of_3'
causal: TRIGGERED (src) | readout: src→dst→src  | gaze: src+0.293 vs dst+0.028 (shifted; clean_prefers_src=False)

causal :
{
    'label': 'src',
    'why': 'The first computation is `n % 3 == 0`, which checks whether n is a multiple of three — sub-goal A.',
    'raw': '<verdict>A</verdict>\n<why>The first computation is `n % 3 == 0`, which checks whether n is a multiple 
of three — sub-goal A.</why>',
    'triggered': True
}

readout:
{
    'source': 'src',
    'dest': 'dst',
    'patched': 'src',
    'valid': True,
    'shifted': True,
    'details': {
        'source': {
            'label': 'src',
            'why': 'The phrase explicitly states checking divisibility by 3, which is sub-goal (A).',
            'raw': '<verdict>A</verdict>\n<why>The phrase explicitly states checking divisibility by 3, which is 
sub-goal (A).</why>'
        },
        'dest': {
            'label': 'dst',
            'why': 'The phrase explicitly states "calculating the sum of the digits," which directly corresponds to
sub-goal (B).',
            'raw': '<verdict>B</verdict>\n<why>The phrase explicitly states "calculating the sum of the digits," 
which directly corresponds to sub-goal (B).</why>'
        },
        'patched': {
            'label': 'src',
            'why': 'The phrase explicitly states checking divisibility by 3, which is sub-goal (A).',
            'raw': '<verdict>A</verdict>\n<why>The phrase explicitly states checking divisibility by 3, which is 
sub-goal (A).</why>'
        }
    }
}

gaze   :
{
    'src_patched': 0.293017681222409,
    'dst_patched': 0.027640122547745706,
    'shifted': True,
    'margin': 0.2653775586746633,
    'src_clean': 0.06555519555695355,
    'dst_clean': 0.252705255150795,
    'clean_prefers_src': False
}

: 

---
## Composed-filter FAILURE (`filter_transfer_setups`)

Same as the composed-filter section of `04_q_transfer.ipynb`, seeded on a causal failure from the completed **filter-transfer** run. The current `materialize` fix (pivot on `if`, not the dangling space) applies here — some of these flip to PASS on re-run.

In [19]:
ffails = load_fails(F_ROOT)
fcands = show_fails(ffails)

450 causal failures  (382 readout-shifted — prioritized)
  [filter__list_str__targeted__python-python] SRC 'palindrome' -> DST 'all_upper' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'all_upper' -> DST 'capitalized' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'same_reversed' -> DST 'all_upper' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'starts_a' -> DST 'all_upper' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'short' -> DST 'all_upper' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'all_lower' -> DST 'all_alpha' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'capitalized' -> DST 'all_alpha' | causal=dst · readout_patched=src
  [filter__list_str__targeted__python-python] SRC 'length_five' -> DST 'all_upper' | causal=dst · readout_patched=both
  [filte

In [38]:
# select the SPECIFIC concept pair `SRC.target <- DST.frontier` (matches the scale-up `pair` column)
# TARGET, FRONTIER = "long", "last_is_digit"        # the long<-last_is_digit case (failed all 4 strategies)
# TARGET, FRONTIER = "starts_a", "all_lower"
TARGET, FRONTIER = "long", "all_upper"
finst = make_filter("list[str]", TARGET, FRONTIER)
fpick = pick_by_pair(ffails, TARGET, FRONTIER)    # its recorded batch-run failure (for the OLD verdict), if any
# fpick = pick_failure(ffails, i=18); finst = rebuild(fpick)   # <- or pick a recorded failure by index

fm = D.materialize(finst)
fsrc, fdst = fm["src_prompt"], fm["dst_prompt"]
fpiv_src, fpiv_dst = pivot_of(fsrc), pivot_of(fdst)
print(f"selected  SRC.target={finst.expected_next!r}  <-  DST.frontier={finst.meta['dst_next']!r}  ({finst.expected_next_desc})")
if fpick:
    print(f"OLD batch verdict: causal={fpick['causal']['label']} (FAILED) · readout_patched={fpick.get('readout', {}).get('patched')}")
show(fsrc, f"SRC · prefilled {finst.src_fill!r} · about to apply {finst.expected_next!r} · pivot@{fpiv_src}",
     emph_span=token_span(fsrc, fm["task_text"]), piv=-1)
show(fdst, f"DST · prefilled {finst.dst_fill!r} · about to apply {finst.meta['dst_next']!r} · pivot@{fpiv_dst}",
     emph_span=token_span(fdst, fm["task_text"]), piv=-1)

selected  SRC.target='long'  <-  DST.frontier='all_upper'  (are longer than three characters)
OLD batch verdict: causal=dst (FAILED) · readout_patched=src


╭─ SRC · prefilled 'all_upper' · about to apply 'long' · pivot@119 ───────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ all the words in `words` that are all uppercase letters and are longer than three characters\n                  │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [w for w in words if w.isupper()]\n                                                                   │
│     return [w for w in res if                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · prefilled 'long' · about to apply 'all_upper' · pivot@122 ───────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ all the words in `words` that are all uppercase letters and are longer than three characters\n                  │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [w for w in words if len(w) > 3]\n                                                                    │
│     return [w for w in res if                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Intervention — same query transplant, TASK span = the composed phrase

Reuses `q_inject_specs` / `Q_LAYERS` from the q-transfer intervention above; the only change is `TASK_POS = token_span(fdst, m["task_text"])`.

In [39]:
PATCH_HEADS = GOAL_HEADS  # heads carrying the transplant (GOAL | UNION | ...)
len(PATCH_HEADS)

122

In [40]:
# ── composed-filter intervention: transplant SRC pivot query into DST — grouped per run ──
from src.attention import get_attention_matrices
Q_LAYERS  = sorted({l for l, _ in PATCH_HEADS})
fq_src    = cache_src_proj(fsrc, [fpiv_src], Q_LAYERS, "q")
fsrc_tok  = prepare_input(prompts=fsrc, tokenizer=mt)
fdst_tok  = prepare_input(prompts=fdst, tokenizer=mt)
fpf, fdc  = q_inject_specs(fq_src, fpiv_dst)              # SAME machinery as the q-transfer cell

# TASK span = the single COMPOSED phrase (FilterTask has .conditions, not a per-line list); query = pivot
TASK_POS  = list(range(*token_span(fdst, fm["task_text"])))
task_toks = [mt.tokenizer.decode([int(fdst_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_i  = get_attention_matrices(fdst_tok, mt, value_weighted=True)
def _cattn(h): return pivot_attn(_clean_i, list(h), TASK_POS, fpiv_dst)

# ── source_run — clean SRC (frontier = expected_next) ──
fgen_src = greedy(fsrc_tok, max_new=64)
rich_print(f"[red1]source_run · clean SRC (frontier {finst.expected_next!r}):[/red1] \n{escape(fgen_src)}")
fro_src = goalscope_readout("   ↳ goalscope · source_run", fsrc, fpiv_src, heads=PATCH_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST (frontier = dst_next); ATTN_VIS baseline ──
fgen_dst = greedy(fdst_tok, max_new=64)
rich_print(f"[sea_green3]destination_run · clean DST (frontier {finst.meta['dst_next']!r}):[/sea_green3] \n{escape(fgen_dst)}")
fro_dst = goalscope_readout("   ↳ goalscope · destination_run", fdst, fpiv_dst, heads=PATCH_HEADS)[0]["patched_gen"]
fattn_clean_goal = _cattn(GOAL_HEADS)      # clean baseline for the gaze eval
show_delta_tokens(task_toks, _cattn(PATCH_HEADS), "ATTN_VIS · pivot→TASK (clean · PATCH_HEADS)")
# show_delta_tokens(task_toks, _cattn(GOAL_HEADS),  "ATTN_VIS · pivot→TASK (clean · GOAL_HEADS)")

# ── patched_run — SRC query injected into DST (expect expected_next); ATTN_DIFF ──
_iface = make_patch_interface(mt, fpf)
fgen_patched = greedy(fdst_tok, prefill_specs=fpf, decode_specs=fdc, n_steer=32, max_new=64)
rich_print(f"[salmon1]patched_run · DST (expect {finst.expected_next!r}):[/salmon1] \n{escape(fgen_patched)}")
fro_patched = goalscope_readout("   ↳ goalscope · patched_run", fdst, fpiv_dst, heads=PATCH_HEADS, iface=_iface)[0]["patched_gen"]
_patched_i = get_attention_matrices(fdst_tok, mt, value_weighted=True, patch_interface=_iface)
fattn_patched_goal = pivot_attn(_patched_i, list(GOAL_HEADS), TASK_POS, fpiv_dst)   # patched, for gaze
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(PATCH_HEADS), TASK_POS, fpiv_dst) - _cattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(task_toks, fattn_patched_goal, "ATTN_VIS · patched (GOAL_HEADS)")

source_run · clean SRC (frontier 'long'): 
 len(w) > 3]
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=119 ' if' => I'm checking if the length of the string is greater than 4.

destination_run · clean DST (frontier 'all_upper'): 
 w.isupper()]

# Example usage:
# words = ["HELLO", "WORLD", "hi", "PYTHON", "CODE", "JAVA"]
# print(func(words))  # Output: ['HELLO', 'WORLD', 'PYTHON', 'JAVA']
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination_run  pos=122 ' if' => I'm checking if the string is uppercase.

... (2 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=122 ' if' => I'm checking if the string is longer than 4 characters and consists 
only of uppercase letters.

ATTN_DIFF · patched−clean (PATCH_HEADS)   (green +Δ / red −Δ, |Δ|max=0.857)

ATTN_VIS · patched (GOAL_HEADS)   (green +Δ / red −Δ, |Δ|max=0.621)

In [43]:
# ── DEBUG VARIANTS on the composed-filter causal intervention ──────────────────
# Reuses fsrc/fdst, fpiv_src/fpiv_dst, fsrc_tok/fdst_tok, PATCH_HEADS, fpf/fdc from the cell above.
# (1) boost the PATCH_HEADS OV write by ALPHA after the q-patch; (2) transplant the SRC query over the
# last N_PIVOT pivot tokens on the PREFILL (targeted postfix -> those tokens match SRC/DST); decode still
# steers the last token only (n_steer). Baseline for comparison: fgen_patched from the cell above.
ALPHA   = 2.0     # (1) amplify how much PATCH_HEADS write into the residual at the decision point
N_PIVOT = 5       # (2) # of trailing pivot tokens whose Q is transplanted on the prefill
N_STEER = 32

rich_print(f"[grey62]baseline · single-pivot q-patch:[/grey62] \n{escape(fgen_patched)}")

# ── (1) boost PATCH_HEADS OV contribution after the q-patch ──────────────────────
def make_boosted_interface(mt, patch_specs, boost_heads, alpha, positions=(-1,)):
    """goalscope/attn patch_interface: q-patch AND boost the boost_heads' OV write (composed)."""
    def iface(mt, inputs, patches=None, model_kwargs=None):
        with P.proj_patch_hooks(mt, patch_specs), P.boost_head_ov(mt, boost_heads, alpha, positions=positions):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return iface

with P.boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)):
    fgen_boost = greedy(fdst_tok, prefill_specs=fpf, decode_specs=fdc, n_steer=N_STEER, max_new=64)
rich_print(f"[salmon1]patched + OV-boost(α={ALPHA}) · expect {finst.expected_next!r}:[/salmon1] \n{escape(fgen_boost)}")
_iface_boost = make_boosted_interface(mt, fpf, PATCH_HEADS, ALPHA)
fro_boost = goalscope_readout("   ↳ goalscope · patched+boost", fdst, fpiv_dst, heads=PATCH_HEADS, iface=_iface_boost)[0]["patched_gen"]

# ── (2) transplant SRC Q over the last N_PIVOT pivot tokens (prefill); decode still patches -1 ──
def q_inject_specs_multi(src_prompt, src_pivot, dst_pivot, heads, n_pivot):
    qh = P.heads_per_layer(heads, "q")
    q_layers = sorted({l for l, _ in heads})
    src_win = list(range(src_pivot - n_pivot + 1, src_pivot + 1))   # cache row r -> dst_win[r]
    dst_win = list(range(dst_pivot - n_pivot + 1, dst_pivot + 1))
    qcache  = cache_src_proj(src_prompt, src_win, q_layers, "q")     # (n_pivot, width) per layer
    prefill = [dict(which="q", cache=qcache, heads=qh, positions=dst_win)]
    dcache  = {l: qcache[l][-1:] for l in qcache}                    # pivot only (last row) for decode
    decode  = [dict(which="q", cache=dcache, heads=qh, positions=[-1])]
    return prefill, decode

# sanity: the last N_PIVOT tokens should coincide on SRC and DST (targeted postfix)
_sw = [mt.tokenizer.decode([int(fsrc_tok["input_ids"][0][fpiv_src - N_PIVOT + 1 + i])]) for i in range(N_PIVOT)]
_dw = [mt.tokenizer.decode([int(fdst_tok["input_ids"][0][fpiv_dst - N_PIVOT + 1 + i])]) for i in range(N_PIVOT)]
print(f"pivot window (N_PIVOT={N_PIVOT})  SRC {_sw}  ->  DST {_dw}  {'✓ aligned' if _sw == _dw else '✗ MISMATCH'}")

fpf_m, fdc_m = q_inject_specs_multi(fsrc, fpiv_src, fpiv_dst, PATCH_HEADS, N_PIVOT)
fgen_multi = greedy(fdst_tok, prefill_specs=fpf_m, decode_specs=fdc_m, n_steer=N_STEER, max_new=64)
rich_print(f"[salmon1]patched · Q over last {N_PIVOT} pivots · expect {finst.expected_next!r}:[/salmon1] \n{escape(fgen_multi)}")
_iface_multi = make_patch_interface(mt, fpf_m)
fro_multi = goalscope_readout("   ↳ goalscope · multi-pivot", fdst, fpiv_dst, heads=PATCH_HEADS, iface=_iface_multi)[0]["patched_gen"]

# ── (1)+(2) combined — multi-pivot Q transplant + OV boost ──────────────────────
with P.boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)):
    fgen_both = greedy(fdst_tok, prefill_specs=fpf_m, decode_specs=fdc_m, n_steer=N_STEER, max_new=64)
rich_print(f"[salmon1]patched · multi-pivot + OV-boost(α={ALPHA}):[/salmon1] \n{escape(fgen_both)}")

baseline · single-pivot q-patch: 
 w.isupper()]

</code>

patched + OV-boost(α=2.0) · expect 'long': 
 w.isupper()]
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched+boost  pos=122 ' if' => I'm checking if the string is longer than 4 characters and 
consists only of uppercase letters.

pivot window (N_PIVOT=5)  SRC [' for', ' w', ' in', ' res', ' if']  ->  DST [' for', ' w', ' in', ' res', ' if']  ✓ aligned


patched · Q over last 5 pivots · expect 'long': 
 w.isupper()]

</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · multi-pivot  pos=122 ' if' => I'm checking if the string is longer than 4 characters and consists 
only of uppercase letters.

patched · multi-pivot + OV-boost(α=2.0): 
 w.isupper()]
</code>

In [31]:
print(fgen_boost)

 w[-1].isalpha() and len(w) > 3 and len(w) > 4]
</code>


### Evaluate the composed-filter intervention

Same `eval_q_transfer` (the filter instance exposes the same `expected_next_desc` / `dst_next_desc` / `task_desc`), on the filter run's captured outputs.

In [32]:
feval = E.eval_q_transfer(
    finst,
    patched_gen=fgen_boost,                                    # 1. causal (oracle)
    readouts=(fro_src, fro_dst, fro_patched),                    # 2. readout shift (oracle)
    gaze=dict(prompt=fdst, task_positions=TASK_POS,              # 3. gaze (rule-based, absolute)
              attn_patched=fattn_patched_goal, attn_clean=fattn_clean_goal),
)
rich_print(f"[bold]EVAL (filter)[/bold] · expected_next={finst.expected_next!r}\n{feval.summary()}")
rich_print("  causal :", feval.causal)
rich_print("  readout:", feval.readout)
rich_print("  gaze   :", feval.gaze)

EVAL (filter) · expected_next='long'
causal: no (dst) | readout: src→dst→src  | gaze: src+0.506 vs dst+0.148 (shifted; clean_prefers_src=False)

causal :
{
    'label': 'dst',
    'why': 'The first computation `w[-1].isalpha()` examines the last character of the word, which is an attempt 
(even if incorrectly using `.isalpha()` instead of `.isdigit()`) to check whether the word ends with a digit — 
sub-goal B.',
    'raw': 'The first computation is `w[-1].isalpha()` — this checks if the last character is alphabetic (which is 
the negation/complement of checking if it ends with a digit, but actually it\'s checking `.isalpha()` not 
`.isdigit()`). Wait, let me re-read.\n\n`w[-1].isalpha()` checks if the last character is alphabetic. This is NOT 
checking if it ends with a digit (that would be `w[-1].isdigit()`). But it\'s related to the "ends with a digit" 
concept (it\'s the wrong implementation of it, but it\'s still attempting to filter based on the last 
character).\n\nActually, the question asks me to judge what sub-goal the computation pursues based on the actual 
logic. The code `w[-1].isalpha()` checks whether the last character is a letter — this is an attempt (albeit 
incorrect) to check something about the last character. The intent seems to be checking if the last character is 
NOT alphabetic (i.e., is a digit), but the actual computation checks if it IS alphabetic.\n\nRegardless of 
correctness, the computation is examining `w[-1]` (the last character) and calling `.isalpha()` on it — this is 
clearly oriented toward sub-goal (B): "end with a digit" (even if implemented incorrectly).\n\nThe `len(w) > 3` 
part comes after and pursues sub-goal (A).\n\nSince the FIRST computation is `w[-1].isalpha()`, which is about the 
last character (pursuing B), and then `len(w) > 3` pursues A...\n\nBut I need to judge ONLY the FIRST 
computation/statement. The first thing evaluated in a chained `and` expression is `w[-1].isalpha()`, which pursues 
(B).\n\n<verdict>B</verdict><why>The first computation `w[-1].isalpha()` examines the last character of the word, 
which is an attempt (even if incorrectly using `.isalpha()` instead of `.isdigit()`) to check whether the word ends
with a digit — sub-goal B.</why>',
    'triggered': False
}

readout:
{
    'source': 'src',
    'dest': 'dst',
    'patched': 'src',
    'valid': True,
    'shifted': True,
    'details': {
        'source': {
            'label': 'src',
            'why': 'The phrase describes a string length comparison, which corresponds to sub-goal (A) "are longer 
than three characters," even though the threshold (>4 vs >3) may be slightly off.',
            'raw': 'The phrase says "checking if the length of the string is greater than 4." This is a length 
check. Note that "longer than three characters" means length > 3, which is equivalent to length >= 4. Checking 
length > 4 would mean length >= 5, which is "longer than four characters," not "longer than three characters." 
However, this is still a length/size check that relates to sub-goal (A) rather than sub-goal (B) which is about 
ending with a digit.\n\n<verdict>A</verdict>\n<why>The phrase describes a string length comparison, which 
corresponds to sub-goal (A) "are longer than three characters," even though the threshold (>4 vs >3) may be 
slightly off.</why>'
        },
        'dest': {
            'label': 'dst',
            'why': 'The phrase explicitly describes checking if the last character is a digit, which directly 
implements the "ends with a digit" condition (sub-goal B).',
            'raw': '<verdict>B</verdict><why>The phrase explicitly describes checking if the last character is a 
digit, which directly implements the "ends with a digit" condition (sub-goal B).</why>'
        },
        'patched': {
            'label': 'src',
            'why': 'The phrase explicitly checks length > 4 (pursuing sub-goal A: longer than three characters), 
while the alphabetic check is unrelated to either sub-goal (and actually contradicts B).',
            'raw': 'The phrase mentions checking if the length is greater than 4 (which relates to "longer than 
three characters" - sub-goal A) and if the string consists only of alphabetic characters (which is unrelated to 
either sub-goal, as it\'s checking for alphabetic characters rather than ending with a digit).\n\nThe length check 
pursues sub-goal A, but the alphabetic check is neither A nor B (it\'s actually the opposite of B - checking for 
alphabetic means NOT ending with a digit).\n\n<verdict>A</verdict><why>The phrase explicitly checks length > 4 
(pursuing sub-goal A: longer than three characters), while the alphabetic check is unrelated to either sub-goal 
(and actually contradicts B).</why>'
        }
    }
}

gaze   :
{
    'src_patched': 0.5064435511827469,
    'dst_patched': 0.14838860370218754,
    'shifted': True,
    'margin': 0.3580549474805593,
    'src_clean': 0.09991892129182815,
    'dst_clean': 0.5752674955874681,
    'clean_prefers_src': False
}

---
## Scale-up — compare 4 intervention strategies (composed-filter, N=16)

Sample N compatible composed-filter instances (current `FILTER_INCOMPATIBLE` rules; **not** restricted to failures) and run each under four query-transplant strategies, then the full eval pipeline (causal / readout-shift / gaze):

1. **last-tok** — single-pivot query patch (current/simplest).
2. **last-tok + boost** — same, but boost the PATCH_HEADS OV write by `ALPHA` at the decision token.
3. **last-5** — transplant the query over the last `N_PIVOT` pivot tokens (prefill); decode still steers the last token only.
4. **last-5 + boost** — (3) + OV boost.

Clean SRC/DST readouts + clean gaze are computed once per instance (shared); only the patched run differs per strategy. _This runs 16×4 interventions — a few minutes on GPU._

In [12]:
# ── SCALE-UP: run the 4 strategies over N composed-filter instances ──
import time, pandas as pd

N        = 16
DATATYPE = "list[str]"
ALPHA    = 2.0
N_PIVOT  = 5
N_STEER  = 32
MAX_NEW  = 64
GAZE_HEADS = list(GOAL_HEADS)     # heads for the gaze/attn readout; PATCH_HEADS carries the transplant

STRATEGIES = {
    "1·last-tok":       dict(n_pivot=1,       alpha=1.0),
    "2·last-tok+boost": dict(n_pivot=1,       alpha=ALPHA),
    "3·last5":          dict(n_pivot=N_PIVOT, alpha=1.0),
    "4·last5+boost":    dict(n_pivot=N_PIVOT, alpha=ALPHA),
}

def _quiet_readout(prompt, pivot, heads, iface=None):
    r = P.apply_goalscope(prompt=prompt, mt=mt, selected_heads=heads, strategy="additive_norm",
        alpha=0.1, steer_layer_range=(35, 46), batch_size=1, max_new_tokens=128,
        goalscope_prompt=getattr(P, "gsp", None), token_positions=[pivot],
        include_noncode=False, show=False, patch_interface=iface)
    return r[0]["patched_gen"]

def _boosted_iface(mt, specs, heads, alpha, positions=(-1,)):
    def iface(mt, inputs, patches=None, model_kwargs=None):
        with P.proj_patch_hooks(mt, specs), P.boost_head_ov(mt, heads, alpha, positions=positions):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return iface

def _q_specs(src_prompt, s_piv, d_piv, heads, n_pivot):
    qh = P.heads_per_layer(heads, "q"); q_layers = sorted({l for l, _ in heads})
    if n_pivot == 1:
        qc = cache_src_proj(src_prompt, [s_piv], q_layers, "q")
        return ([dict(which="q", cache=qc, heads=qh, positions=[d_piv])],
                [dict(which="q", cache=qc, heads=qh, positions=[-1])])
    s_win = list(range(s_piv - n_pivot + 1, s_piv + 1))
    d_win = list(range(d_piv - n_pivot + 1, d_piv + 1))
    qc = cache_src_proj(src_prompt, s_win, q_layers, "q")
    dc = {l: qc[l][-1:] for l in qc}                 # pivot-only cache for the decode steer
    return ([dict(which="q", cache=qc, heads=qh, positions=d_win)],
            [dict(which="q", cache=dc, heads=qh, positions=[-1])])

set_seed(0)
insts = D.filter_transfer_setups(DATATYPE, n_samples=N, seed=0, postfix="targeted",
                                 src_lang="python", dst_lang="python")
print(f"{len(insts)} filter instances · datatype={DATATYPE} · PATCH_HEADS={len(PATCH_HEADS)} heads · "
      f"strategies={list(STRATEGIES)}")

rows, t0 = [], time.time()
for qi, inst in enumerate(insts):
    m = D.materialize(inst)
    fsrc, fdst = m["src_prompt"], m["dst_prompt"]
    fsrc_tok = prepare_input(prompts=fsrc, tokenizer=mt)
    fdst_tok = prepare_input(prompts=fdst, tokenizer=mt)
    s_piv, d_piv = pivot_of(fsrc), pivot_of(fdst)
    TASK_POS = list(range(*token_span(fdst, m["task_text"])))
    clean_i    = get_attention_matrices(fdst_tok, mt, value_weighted=True)
    attn_clean = pivot_attn(clean_i, GAZE_HEADS, TASK_POS, d_piv)      # shared clean gaze baseline
    ro_src = _quiet_readout(fsrc, s_piv, PATCH_HEADS)                  # shared clean readouts
    ro_dst = _quiet_readout(fdst, d_piv, PATCH_HEADS)
    for name, cfg in STRATEGIES.items():
        pf, dc = _q_specs(fsrc, s_piv, d_piv, PATCH_HEADS, cfg["n_pivot"])
        plain_iface = make_patch_interface(mt, pf)
        if cfg["alpha"] != 1.0:
            with P.boost_head_ov(mt, PATCH_HEADS, cfg["alpha"], positions=(-1,)):
                gen = greedy(fdst_tok, prefill_specs=pf, decode_specs=dc, n_steer=N_STEER, max_new=MAX_NEW)
            ro_iface = _boosted_iface(mt, pf, PATCH_HEADS, cfg["alpha"])
        else:
            gen = greedy(fdst_tok, prefill_specs=pf, decode_specs=dc, n_steer=N_STEER, max_new=MAX_NEW)
            ro_iface = plain_iface
        ro_pat   = _quiet_readout(fdst, d_piv, PATCH_HEADS, iface=ro_iface)
        pat_i    = get_attention_matrices(fdst_tok, mt, value_weighted=True, patch_interface=plain_iface)
        attn_pat = pivot_attn(pat_i, GAZE_HEADS, TASK_POS, d_piv)      # boost doesn't move attention -> plain iface
        ev = E.eval_q_transfer(inst, patched_gen=gen, readouts=(ro_src, ro_dst, ro_pat),
            gaze=dict(prompt=fdst, task_positions=TASK_POS, attn_patched=attn_pat, attn_clean=attn_clean))
        rows.append(dict(inst=qi, pair=f"{inst.expected_next}<-{inst.meta['dst_next']}", strategy=name,
            causal=bool(ev.causal["triggered"]), causal_label=ev.causal["label"],
            ro_valid=ev.readout["valid"], ro_shifted=ev.readout["shifted"],
            gaze=ev.gaze["shifted"], gen=gen[:60]))
    print(f"  [{qi+1}/{len(insts)}] {inst.expected_next!r}<-{inst.meta['dst_next']!r} ({time.time()-t0:.0f}s)")

resdf = pd.DataFrame(rows)
print(f"\n{len(resdf)} rows ({len(insts)} insts x {len(STRATEGIES)} strategies)")

16 filter instances · datatype=list[str] · PATCH_HEADS=228 heads · strategies=['1·last-tok', '2·last-tok+boost', '3·last5', '4·last5+boost']


GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

... (103 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

  [16/16] 'long'<-'all_upper' (1473s)

64 rows (16 insts x 4 strategies)


In [13]:
# ── compare the 4 strategies ──
def _rate(s):
    s = s.dropna(); return s.mean() if len(s) else float("nan")

summary = []
for name in STRATEGIES:
    sub = resdf[resdf.strategy == name]
    v = sub[sub.ro_valid == True]
    summary.append(dict(strategy=name, n=len(sub), causal=_rate(sub.causal),
                        readout_shift_valid=_rate(v.ro_shifted), gaze=_rate(sub.gaze)))
S = pd.DataFrame(summary).set_index("strategy")
print(f"=== strategy comparison · composed-filter · N={resdf.inst.nunique()} ===")
display(S.style.format({"causal": "{:.1%}", "readout_shift_valid": "{:.1%}", "gaze": "{:.1%}", "n": "{:.0f}"}))

# per-instance causal HIT/miss matrix — which strategy rescues which instance
piv = resdf.pivot_table(index=["inst", "pair"], columns="strategy", values="causal", aggfunc="first")
display(piv.replace({True: "✅", False: "·"}))

# net gain/loss vs strategy #1 (does boost / multi-pivot rescue more than it breaks?)
base = resdf[resdf.strategy == "1·last-tok"].set_index("inst").causal
for name in STRATEGIES:
    cur = resdf[resdf.strategy == name].set_index("inst").causal
    gained = int((cur & ~base).sum()); lost = int((~cur & base).sum())
    print(f"  {name:18} causal={cur.mean():.1%}   vs #1:  +{gained} rescued / -{lost} broken")

=== strategy comparison · composed-filter · N=16 ===


,n,causal,readout_shift_valid,gaze
strategy,,,,
1·last-tok,16,31.2%,92.9%,100.0%
2·last-tok+boost,16,62.5%,92.9%,100.0%
3·last5,16,50.0%,92.9%,100.0%
4·last5+boost,16,75.0%,92.9%,100.0%


,strategy,1·last-tok,2·last-tok+boost,3·last5,4·last5+boost
inst,pair,,,,
0,starts_a<-same_reversed,✅,✅,✅,✅
1,vowel_start<-all_lower,·,✅,·,✅
2,same_reversed<-short,✅,✅,✅,✅
3,length_five<-all_alpha,·,·,·,·
4,contains_digit<-short,·,✅,✅,✅
5,starts_a<-all_lower,·,·,·,·
6,same_reversed<-all_lower,·,·,·,✅
7,long<-last_is_digit,·,·,·,·
8,ends_s<-length_five,·,✅,·,✅


  1·last-tok         causal=31.2%   vs #1:  +0 rescued / -0 broken
  2·last-tok+boost   causal=62.5%   vs #1:  +5 rescued / -0 broken
  3·last5            causal=50.0%   vs #1:  +3 rescued / -0 broken
  4·last5+boost      causal=75.0%   vs #1:  +7 rescued / -0 broken
